In [19]:
import networkx as nx
import numpy as np
import scipy.sparse as sp
import os
import time
from collections import defaultdict
# import tensorflow as tf
import pandas as pd
import pdb
from ast import literal_eval
import copy
import pickle
import sys
sys.path.append(os.path.join(os.path.dirname(sys.path[0]),'sim'))
from params import ENVIRONMENT_BOUNDARY_X, TIME_LIMIT, COMMIT_THRESHOLD
from math import sqrt
# from itertools import compress
from concurrent.futures import ThreadPoolExecutor, as_completed
import csv

In [20]:
def round_function(x, d):
    new = []
    for r in x:
        new.append(np.round(r,decimals=d))
    # pdb.set_trace()
    return tuple(new)

def new_state_from_old(old_):
    # pdb.set_trace()
    # qindices = 
    old = copy.deepcopy(old_)
    # pdb.set_trace()
    nA = int(len(old)/4)
    oldarr = np.reshape(old, (nA, 4)).tolist()
    sites_state = [1.0, 1.0, 0.0]*4
    unique_sites = defaultdict(int)
    # oldarr[:,0] = 0
    # oldarr_unique = np.unique(oldarr, axis=0)
    new = np.zeros((7,6))
    # counter_sites = [0, 0, 0, 0]*7
    # counter = 
    for i in oldarr:
        site = tuple(i[1:])
        unique_sites[site] += 1
    # print('1')
    
    sitelist = [i for i in unique_sites if i!= (1.0, 1.0, 0.0)]
    # quals = [i[2] for i in sitelist]
    # pdb.set_trace()
    sites = sorted(sitelist, key=lambda x: x[2] , reverse=True)
    site_to_id = dict()
    # print('2')
    for id, site in enumerate(sites):
        sites_state[id*3] = site[0]
        sites_state[id*3+1] = site[1]
        sites_state[id*3+2] = site[2]
        site_to_id[site] = id

    # print('3')
    for i in oldarr:
        id_state = int(np.round(6.0*i[0]))
        # print(id_state)
        if id_state >=5 :
            new[id_state,4] += 1.0/nA
        elif id_state == 4:
            new[id_state,5] += 1.0/nA
        else:
            site = tuple(i[1:])
            new[id_state, site_to_id[site]] += 1.0/nA
            if id_state == 0:
                new[id_state,5] += 1.0/nA
            else:
                new[id_state,4] += 1.0/nA


    return round_function(tuple(np.concatenate((np.reshape(new, (1,np.shape(new)[0]*np.shape(new)[1]))[0], sites_state), axis=0).tolist()), 3)

def get_unique_IDs(fl, dict_old, newdict, rd):
    states_unique = np.unique(fl.currentState)
    # states_new = np.zeros(72,)
    # for i in range(len(states_unique)):
    for i in states_unique:
        # pdb.set_trace()
        if i in dict_old:
            continue 
        else:
            dict_old[i] = len(dict_old)
            newstate = new_state_from_old(i)
            rd[newstate] = i
            newdict[newstate] = len(newdict)
    return dict_old, newdict, rd #, nodeSize

def get_edges_success_time(fl, IDLookup, get_edges_with):
    # fl['newState'] = fl.apply(lambda x: rd[x.currentState], axis=1)
    fl['stateIDs'] = fl.apply(lambda x: IDLookup[x.currentState], axis=1)
    # pdb.set_trace()
    for id, csid in enumerate(fl.stateIDs.values):
        
        if id+1 == len(fl.stateIDs.values):
            break
        if csid in get_edges_with:
            get_edges_with[csid].append(fl.stateIDs.values[id+1])
        else:
            get_edges_with[csid] = [fl.stateIDs.values[id+1]]
        # if csid == 88:
        #     pdb.set_trace()


    return get_edges_with

In [21]:
def process_file(file_list, successes, times, quals):
    graph = nx.Graph()
    qrounded = [np.round(q, decimals=3) for q in quals]
    sortedquals = np.sort(list(quals))
    success_vals = []
    for s in successes:
        # print(s, sortedquals[-1])
        success_vals.append(s/sortedquals[-1])
    meanSuccess = np.mean(success_vals)
    meanTime = np.mean(times)
    has_edges_with = dict()
    IDLookup = dict()
    IDLookupnew = dict()
    reversedict = dict()
    for f in file_list:
        fl = pd.read_csv(folder_main + f)
        fl.agent_states = fl.agent_states.apply(literal_eval)
        fl.agent_sites = fl.agent_sites.apply(literal_eval)
        fl.agent_positions = fl.agent_positions.apply(literal_eval)
        fl['currentState'] = fl.node.apply(literal_eval)
        IDLookup, IDLookupnew, reversedict = get_unique_IDs(fl, IDLookup, IDLookupnew, reversedict)
        has_edges_with = get_edges_success_time(fl, IDLookup, has_edges_with)
        
    for nodePos, nodeID in IDLookupnew.items():
        graph.add_node(nodeID, x=nodePos, xold = reversedict[nodePos])

    for node,value in has_edges_with.items():
        for edge_to in value:
            if graph.has_edge(node, edge_to):
                graph[node][edge_to]['weight'] += 1.0
            else:
                graph.add_edge(node, edge_to, weight=1.0)

    nA = len(fl.agent_positions.iloc[0])
    colors = 'b'
    nx.set_node_attributes(graph, nA, 'agents')
    # nx.set_node_attributes(graph, colors, 'colors')
    nx.set_node_attributes(graph, qrounded, 'quals')
    # nx.set_node_attributes(graph, entry[1].iloc[1], 'poses')
    nx.set_node_attributes(graph, meanSuccess, 'success')
    nx.set_node_attributes(graph, meanTime, 'times_conved')
    # nx.set_node_attributes(graph, 0, 'global_info')
    # fl.node = lite
    start_node_id = IDLookup[fl.iloc[0]['currentState']]

    for node in graph.nodes(data=True):
        if node[0] == start_node_id:
            graph.nodes[node[0]]['times_conved'] = meanTime
        else:
            graph.nodes[node[0]]['times_conved'] = -1

    return graph, new_state_from_old(fl.iloc[0]['currentState'])

In [22]:
folder_main = './data/lots_of_node_samples/test_traj_sample_128_easy/'
metadata_file = folder_main + 'metadata.csv'
metadata = pd.read_csv(metadata_file) 
metadata.site_qualities=metadata.site_qualities.apply(literal_eval)
metadata.site_positions=metadata.site_positions.apply(literal_eval)
metadata.site_positions=metadata.site_positions.apply(lambda x: tuple([tuple(a) for a in x]))
metadata.site_qualities=metadata.site_qualities.apply(lambda x: tuple(x))
# metadata.node=metadata.node.apply(literal_eval)
# metadata.sims_train = metadata.sims_train.apply(lambda x: literal_eval(x))
# metadata.site_converged = metadata.site_converged.apply(lambda x: literal_eval(x))
# metadata.time_converged = metadata.time_converged.apply(lambda x: literal_eval(x))
df = metadata.groupby(by=['site_qualities', 'site_positions', 'num_agents', 'node'], as_index=False).agg(lambda x: x.tolist())
# df = metadata.groupby(by=['start_state'], as_index=False).agg(lambda x: x.tolist())

In [23]:
# dict_node_state = dict()
# # list_state = []
# for some_id, entry in enumerate(df.iterrows()):
#     node = entry[1].iloc[3]
#     state = entry[1].iloc[4]
#     if node in dict_node_state:
#         if dict_node_state[node] == state:
#             continue
#         else:
#             print('node: ', node)
#             print('state 1: ', state)
#             print('state 2: ', dict_node_state[node])
#     else:
#         dict_node_state[node] = state


In [24]:
import time
dictnew_time = dict()
dictnew_succ = dict()
dict_files = dict()
folder_graph = folder_main + 'graphs/'
ml = 0
for some_id, entry in enumerate(df.iterrows()):
    times = entry[1].iloc[7]
    ml = len(times) if len(times) > ml else ml

for some_id, entry in enumerate(df.iterrows()):
    # break
    graph_name = str(int(time.time()*1000000))#entry[1].iloc[3]
    graphs_train = entry[1].iloc[4]
    successes = entry[1].iloc[6]
    times = entry[1].iloc[7]
    quals = entry[1].iloc[0]
    if len(times) > 10:
        # print(graphs_train, successes, times, quals)
        graph, node = process_file(graphs_train, successes, times, quals)
        print('number of nodes: ', graph.number_of_nodes())
        # print('number of edges: ', graph.number_of_edges())
        # print('average degree: ', sum(dict(graph.degree()).values()) / graph.number_of_nodes())
        # folder_graph = folder_main + 'graphs/'
        fil = open(folder_graph+str(graph_name)+'.pickle', 'wb')
        pickle.dump(graph, fil)
        fil.close()
        dictnew_time[str(node)] = times + [np.nan] * (ml - len(times))
        dictnew_succ[str(node)] = successes + [np.nan] * (ml - len(times))
        for g in graphs_train:
            dict_files[g] = str(node)

# print(dictnew_time)
dftime = pd.DataFrame.from_dict(dictnew_time)#, orient='index', columns=['file', 'times'])
dfsucc = pd.DataFrame.from_dict(dictnew_succ)#, orient='index', columns=['file', 'succ'])
dffiles = pd.DataFrame.from_dict(dict_files, orient='index', columns=['ref'])

# print(dftime[graph_name+'.pickle'].tolist())
dftime.to_csv(folder_graph+'times.csv')
dfsucc.to_csv(folder_graph+'succ.csv')
dffiles.to_csv(folder_graph+'files.csv')
    # break 
# print(entry)



number of nodes:  742
number of nodes:  729
number of nodes:  781
number of nodes:  1608
number of nodes:  718
number of nodes:  727
number of nodes:  848
number of nodes:  570
number of nodes:  640
number of nodes:  383
number of nodes:  1882
number of nodes:  667
number of nodes:  1633
number of nodes:  1190
number of nodes:  1543
number of nodes:  1144
number of nodes:  1865
number of nodes:  1859
number of nodes:  894
number of nodes:  624
number of nodes:  1546
number of nodes:  1496
number of nodes:  1369
number of nodes:  2135
number of nodes:  2032
number of nodes:  1206
number of nodes:  1016
number of nodes:  2367
number of nodes:  1837
number of nodes:  1530
number of nodes:  1014
number of nodes:  1078
number of nodes:  879
number of nodes:  1024
number of nodes:  2240
number of nodes:  2081
number of nodes:  1632
number of nodes:  1663
number of nodes:  1480
number of nodes:  722
number of nodes:  1654
number of nodes:  798
number of nodes:  1146
number of nodes:  1473
num

In [25]:
# # CHECK GRAPHS
# files = os.listdir(folder_graph)
# files = [file for file in files if file.endswith('.pickle') and file.startswith('1')]
# fil = open(folder_graph + files[0], 'rb')
# G = pickle.load(fil)
# fil.close()
# for node in G.nodes(data=True):
#     print(node)
# # G.nodes[1]